# Model predictions vs experimental data (CMS + ATLAS)

Global $v_0$

In [6]:
#!/usr/bin/env python3
"""
Emulator predictions vs experimental data — Global v0.

Posteriors plotadas:
  - Prior                    → grey hatch band
  - original_posterior       → orange dashed band  (pT-integrated, new_LHC_posterior_samples.csv)
  - posterior (Scaled Spectra) → blue solid band   (posterior/mcmc_chain_{corr}.csv)

Dados:
  - ATLAS 5.02 TeV, 0.5–2 GeV  → open circle, golden
  - CMS   2.76 TeV, 0.5–2 GeV  → filled circle, golden
"""

import struct
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

# =============================================================================
# STYLE
# =============================================================================

plt.rcParams.update({
    "xtick.major.size":     6,    "ytick.major.size":     6,
    "xtick.minor.size":     3,    "ytick.minor.size":     3,
    "xtick.major.width":    1.2,  "ytick.major.width":    1.2,
    "xtick.minor.width":    1.0,  "ytick.minor.width":    1.0,
    "axes.linewidth":       1.2,
    "lines.linewidth":      1.5,
    "xtick.direction":      "in", "ytick.direction":      "in",
    "xtick.minor.visible":  True, "ytick.minor.visible":  True,
    "xtick.major.pad":      4,    "ytick.major.pad":      4,
    "legend.handletextpad": 0.4,
    "text.usetex":          False,
    "font.family":          "serif",
    "font.size":            14,
})

# =============================================================================
# CONFIGURATION
# =============================================================================

EXP_ATLAS_FILE   = "experimental_data/differential_radial_flow/HEPData-ins2907010-v1-Figure_2a.csv"
CMS_MAIN_ROOT    = "main.root"
CMS_SYST_ROOT    = "SystUncs.root"
PREDICTIONS_FILE = "predictions_global_v0/all_predictions_global_v0.pkl"

CORRECTION    = "Grad"
OUTPUT_FOLDER = "predictions_global_v0_vs_exp"

# Colours
ATLAS_COLOR  = "#b8860b"    # dark golden    → open circle
CMS_COLOR    = "#b8860b"    # dark golden → filled square
PRIOR_COLOR  = "grey"
POST1_COLOR  = "darkorange" # pT-integrated  → dashed border
POST2_COLOR  = "#1f6fbd"    # Scaled Spectra → solid band

# ATLAS centrality grid
ATLAS_CENT_EDGES   = [0, 5, 10, 20, 30, 40, 50, 60, 70, 80]
ATLAS_CENT_CENTERS = np.array(
    [(ATLAS_CENT_EDGES[i] + ATLAS_CENT_EDGES[i+1]) / 2.0
     for i in range(len(ATLAS_CENT_EDGES)-1)]
)   # [2.5, 7.5, 15, 25, 35, 45, 55, 65, 75]

Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

# =============================================================================
# CMS ROOT PARSER
# =============================================================================

def _read_pascal(raw, pos):
    n = raw[pos]
    return raw[pos+1: pos+1+n].decode("latin-1", errors="replace"), pos+1+n

def _parse_root_keys(raw):
    keys = {}
    pos  = 100
    while pos < len(raw) - 26:
        nbytes = struct.unpack(">i", raw[pos: pos+4])[0]
        if nbytes == 0:  break
        if nbytes < 0:   pos += -nbytes; continue
        keylen = struct.unpack(">H", raw[pos+14: pos+16])[0]
        sp = pos + 26
        try:
            classname, sp = _read_pascal(raw, sp)
            name,      sp = _read_pascal(raw, sp)
            _,         sp = _read_pascal(raw, sp)
        except Exception:
            pos += nbytes; continue
        keys[name] = {"data_start": pos+keylen,
                      "data_len":   nbytes-keylen}
        pos += nbytes
    return keys

def _rd(data, off):
    return struct.unpack(">d", data[off: off+8])[0]

def _parse_v0_scalar(raw, key):
    data = raw[key["data_start"]: key["data_start"]+key["data_len"]]
    for x_off in range(70, 110):
        if x_off + 8 > len(data): break
        x = _rd(data, x_off)
        if abs(x-55) < 0.1 or abs(x-65) < 0.1:
            y    = _rd(data, x_off+8+1)
            stat = abs(_rd(data, len(data)-8))
            return {"x": x, "y": y, "stat": stat}
    return None

def _parse_v0_syst_graph(raw_s, keys_s, name="gr_v0_total"):
    if name not in keys_s: return None
    k    = keys_s[name]
    data = raw_s[k["data_start"]: k["data_start"]+k["data_len"]]
    for x_off in range(70, 110):
        if x_off+16 > len(data): break
        x0, x1 = _rd(data, x_off), _rd(data, x_off+8)
        if abs(x0-55) < 0.1 and abs(x1-65) < 0.1:
            y_off  = x_off + 16 + 1
            ex_off = y_off  + 16 + 57
            ey_off = ex_off + 16 + 1
            return {"ey": np.array([_rd(data, ey_off), _rd(data, ey_off+8)])}
    return None

def load_cms_v0(main_path, syst_path):
    mp, sp = Path(main_path), Path(syst_path)
    if not mp.exists() or not sp.exists():
        print("  WARNING: CMS ROOT files not found.")
        return None
    raw, raw_s = mp.read_bytes(), sp.read_bytes()
    keys_m = _parse_root_keys(raw)
    keys_s = _parse_root_keys(raw_s)
    r55 = _parse_v0_scalar(raw, keys_m["v0_55_1"])
    r65 = _parse_v0_scalar(raw, keys_m["v0_65_1"])
    if r55 is None or r65 is None:
        print("  WARNING: could not parse CMS scalar objects.")
        return None
    sg   = _parse_v0_syst_graph(raw_s, keys_s)
    syst = sg["ey"] if sg is not None else np.zeros(2)
    cent  = np.array([r55["x"], r65["x"]])
    v0    = np.array([r55["y"], r65["y"]])
    stat  = np.array([r55["stat"], r65["stat"]])
    total = np.sqrt(stat**2 + syst**2)
    print(f"  ✓ CMS 2.76 TeV  cent={cent}  v0={v0.round(5)}  "
          f"stat={stat.round(5)}  syst={syst.round(5)}")
    return {"centrality": cent, "v0": v0,
            "stat": stat, "syst": syst, "total": total}

# =============================================================================
# LOAD ATLAS DATA
# =============================================================================

def load_atlas_v0(filepath):
    if not Path(filepath).exists():
        print(f"  WARNING: ATLAS file not found: {filepath}")
        return None
    data = pd.read_csv(filepath, comment='#')
    for i, row in data.iterrows():
        if i > 0 and str(row.iloc[0]) == 'Centrality [%]':
            data = data.iloc[:i].copy(); break
    data.columns = ['centrality', 'v0', 'stat_plus', 'stat_minus',
                    'sys_plus',   'sys_minus']
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')
    stat  = np.abs(data['stat_plus'].values)
    syst  = np.abs(data['sys_plus'].values)
    total = np.sqrt(stat**2 + syst**2)
    print(f"  ✓ ATLAS 5.02 TeV  {len(data)} bins  "
          f"cent={data['centrality'].values}")
    return {"centrality": data['centrality'].values,
            "v0":  data['v0'].values,
            "stat": stat, "syst": syst, "total": total}

# =============================================================================
# LOAD & ALIGN PREDICTIONS
# =============================================================================

def load_pkl(filepath):
    if filepath is None or not Path(filepath).exists():
        return None
    with open(filepath, 'rb') as f:
        return pickle.load(f)


def align_to_atlas(pred_data, atlas_centers):
    """Interpolate prediction samples onto the ATLAS centrality grid."""
    stored = np.asarray(pred_data['centrality_centers'])
    print(f"    Stored emulator centres : {stored}")
    print(f"    Target ATLAS centres    : {atlas_centers}")

    out = pred_data.copy()
    out['centrality_centers'] = atlas_centers.copy()

    for key in ('prior', 'original_posterior', 'posterior'):
        arr = pred_data.get(key)
        if arr is None: continue
        out[key] = np.row_stack([
            np.interp(atlas_centers, stored, arr[s, :])
            for s in range(arr.shape[0])
        ])
        print(f"    {key:<25s}: {arr.shape} → {out[key].shape}")

    return out

# =============================================================================
# PLOT HELPERS
# =============================================================================

def configure_axis(ax, fs):
    ax.yaxis.set_ticks_position('both')
    ax.xaxis.set_ticks_position('both')
    ax.tick_params(which='both', labelsize=fs)


def fill_band(ax, cent, arr, color, alpha, label=None, hatch=None):
    """5–95% band — no median line."""
    lo, hi = np.percentile(arr, [5, 95], axis=0)
    kw = dict(alpha=alpha, label=label, zorder=2)
    if hatch:
        kw.update(hatch=hatch, edgecolor=color, facecolor=color)
    else:
        kw.update(color=color)
    ax.fill_between(cent, lo, hi, **kw)


def fill_band_dashed(ax, cent, arr, color, alpha, label=None):
    """5–95% band with dashed boundary lines — no median."""
    lo, hi = np.percentile(arr, [5, 95], axis=0)
    ax.fill_between(cent, lo, hi, color=color, alpha=alpha,
                    label=label, zorder=2)
    ax.plot(cent, lo, color=color, lw=1.5, ls='--', alpha=0.9, zorder=3)
    ax.plot(cent, hi, color=color, lw=1.5, ls='--', alpha=0.9, zorder=3)


def fill_band_ratio(ax, cent, arr, denom_v0, denom_cent,
                    color, alpha):
    """5–95% band in ratio space — no median."""
    lo, hi = np.percentile(arr, [5, 95], axis=0)
    lo_i = np.interp(denom_cent, cent, lo) / denom_v0
    hi_i = np.interp(denom_cent, cent, hi) / denom_v0
    ax.fill_between(denom_cent, lo_i, hi_i,
                    color=color, alpha=alpha, zorder=2)


def fill_band_ratio_dashed(ax, cent, arr, denom_v0, denom_cent,
                           color, alpha):
    """5–95% ratio band with dashed boundary — no median."""
    lo, hi = np.percentile(arr, [5, 95], axis=0)
    lo_i = np.interp(denom_cent, cent, lo) / denom_v0
    hi_i = np.interp(denom_cent, cent, hi) / denom_v0
    ax.fill_between(denom_cent, lo_i, hi_i,
                    color=color, alpha=alpha, zorder=2)
    ax.plot(denom_cent, lo_i, color=color, lw=1.5, ls='--', alpha=0.9, zorder=3)
    ax.plot(denom_cent, hi_i, color=color, lw=1.5, ls='--', alpha=0.9, zorder=3)


def syst_box(ax, xc, yc, sy, color, bw=0.35, lw=1.3):
    """Draw a systematic-error rectangle.

    bw : half-width in centrality [%].  0.35 gives a slim visual marker
         (~0.7% wide) that is clearly taller than it is wide when sy ≳ 0.001.
    """
    ax.add_patch(plt.Rectangle(
        (xc-bw, yc-sy), 2*bw, 2*sy,
        lw=lw, edgecolor=color, facecolor='none', zorder=12))

# =============================================================================
# MAIN PLOT
# =============================================================================

def make_plot(pred, atlas, cms, correction):
    """
    pred : aligned predictions dict with keys:
           'prior', 'original_posterior' (pT-int), 'posterior' (Scaled Spectra)
    """
    fs = 18

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(9, 10),
        gridspec_kw={"height_ratios": [3, 1]},
        sharex=True,
    )

    cent = pred['centrality_centers']

    # ── 1. Prior ─────────────────────────────────────────────────────────────
    if pred.get('prior') is not None:
        fill_band(ax_top, cent, pred['prior'],
                  color=PRIOR_COLOR, alpha=0.30,
                  label="Prior 90% C.I.", hatch="///")
        if atlas is not None:
            fill_band_ratio(ax_bot, cent, pred['prior'],
                            atlas['v0'], atlas['centrality'],
                            color=PRIOR_COLOR, alpha=0.25)

    # ── 2. pT-integrated posterior — orange dashed border ────────────────────
    if pred.get('original_posterior') is not None:
        fill_band_dashed(ax_top, cent, pred['original_posterior'],
                         color=POST1_COLOR, alpha=0.35,
                         label=r"Posterior 90% C.I. ($p_T$-integrated)")
        if atlas is not None:
            fill_band_ratio_dashed(ax_bot, cent, pred['original_posterior'],
                                   atlas['v0'], atlas['centrality'],
                                   color=POST1_COLOR, alpha=0.30)

    # ── 3. Scaled Spectra posterior — blue solid band ─────────────────────────
    if pred.get('posterior') is not None:
        fill_band(ax_top, cent, pred['posterior'],
                  color=POST2_COLOR, alpha=0.40,
                  label=r"Posterior 90% C.I. (Scaled Spectra)")
        if atlas is not None:
            fill_band_ratio(ax_bot, cent, pred['posterior'],
                            atlas['v0'], atlas['centrality'],
                            color=POST2_COLOR, alpha=0.35)

    # ── 4. ATLAS data ─────────────────────────────────────────────────────────
    if atlas is not None:
        ax_top.errorbar(
            atlas['centrality'], atlas['v0'],
            yerr=atlas['stat'],
            fmt='s', mfc='none', color=ATLAS_COLOR,
            markersize=8, capsize=4, lw=1.8, zorder=10,
            label=r"ATLAS 5.02 TeV  ($0.5\!-\!2$ GeV/$c$)",
        )
        for xc, yc, sy in zip(atlas['centrality'], atlas['v0'], atlas['syst']):
            syst_box(ax_top, xc, yc, sy, ATLAS_COLOR)

        ax_bot.errorbar(
            atlas['centrality'],
            np.ones_like(atlas['centrality']),
            yerr=atlas['total'] / atlas['v0'],
            fmt='s', mfc='none', color=ATLAS_COLOR,
            markersize=8, capsize=4, lw=1.8, zorder=10,
        )
        for xc, sy, yv in zip(atlas['centrality'], atlas['syst'], atlas['v0']):
            syst_box(ax_bot, xc, 1.0, sy/yv, ATLAS_COLOR)

    # ── 5. CMS data ───────────────────────────────────────────────────────────
    if cms is not None:
        ax_top.errorbar(
            cms['centrality'], cms['v0'],
            yerr=cms['stat'],
            fmt='s', mfc=CMS_COLOR, color=CMS_COLOR,
            markersize=8, capsize=4, lw=1.8, zorder=10,
            label=r"CMS 2.76 TeV  ($0.5\!-\!2$ GeV/$c$)",
        )
        for xc, yc, sy in zip(cms['centrality'], cms['v0'], cms['syst']):
            syst_box(ax_top, xc, yc, sy, CMS_COLOR)

        if atlas is not None:
            at_cms    = np.interp(cms['centrality'],
                                  atlas['centrality'], atlas['v0'])
            err_cms   = np.interp(cms['centrality'],
                                  atlas['centrality'], atlas['total'])
            ratio_cms = cms['v0'] / at_cms
            rerr_cms  = ratio_cms * np.sqrt(
                (cms['total'] / cms['v0'])**2 +
                (err_cms      / at_cms)**2
            )
            ax_bot.errorbar(
                cms['centrality'], ratio_cms,
                yerr=rerr_cms,
                fmt='s', mfc=CMS_COLOR, color=CMS_COLOR,
                markersize=8, capsize=4, lw=1.8, zorder=10,
                label=r"CMS$\,/\,$ATLAS",
            )

    # ── Formatting ────────────────────────────────────────────────────────────
    if atlas is not None:
        x_lo = atlas['centrality'].min() - 2
        x_hi = atlas['centrality'].max() + 2
    else:
        x_lo, x_hi = 0, 80

    ax_top.set_xlim(x_lo, x_hi)
    ax_top.set_ylim(0, 0.08)
    ax_top.set_ylabel(r"$v_0$", fontsize=fs+8)
    ax_top.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
    ax_top.legend(loc='upper left', frameon=False, fontsize=fs-5)
    ax_top.text(0.97, 0.03,
                r"Pb+Pb,  $p_T^{\rm ref} = 0.5\!-\!2$ GeV/$c$",
                transform=ax_top.transAxes,
                va='bottom', ha='right', fontsize=fs-4)
    configure_axis(ax_top, fs-2)

    ax_bot.axhline(1.0, color='k', ls='--', lw=1.5)
    ax_bot.set_xlim(x_lo, x_hi)
    ax_bot.set_ylim(0.4, 2.8)
    ax_bot.set_xlabel("Centrality [%]", fontsize=fs+4)
    ax_bot.set_ylabel("Model / ATLAS", fontsize=fs+2)
    ax_bot.grid(True, alpha=0.25, which='major')
    handles, labels = ax_bot.get_legend_handles_labels()
    if handles:
        ax_bot.legend(handles, labels, fontsize=fs-6,
                      loc='upper right', frameon=False)
    configure_axis(ax_bot, fs-2)

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.05)
    return fig

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    print("\n" + "="*70)
    print("EMULATOR PREDICTIONS vs EXPERIMENTAL DATA  —  Global v0")
    print(f"Correction: {CORRECTION}  |  pT ref: 0.5–2 GeV/c")
    print("="*70)

    print("\nLoading ATLAS 5.02 TeV data...")
    atlas = load_atlas_v0(EXP_ATLAS_FILE)

    print("\nLoading CMS 2.76 TeV data (ROOT)...")
    cms = load_cms_v0(CMS_MAIN_ROOT, CMS_SYST_ROOT)

    print("\nLoading emulator predictions...")
    all_pred = load_pkl(PREDICTIONS_FILE)
    if all_pred is None or CORRECTION not in all_pred:
        raise FileNotFoundError(
            f"'{CORRECTION}' not found in {PREDICTIONS_FILE}. "
            f"Available: {list(all_pred.keys()) if all_pred else 'file missing'}"
        )

    raw = all_pred[CORRECTION]
    print(f"  Keys in prediction dict: {[k for k in raw if raw[k] is not None]}")
    for key in ('prior', 'original_posterior', 'posterior'):
        arr = raw.get(key)
        if arr is not None:
            print(f"  {key:<25s}: {arr.shape}")
        else:
            print(f"  {key:<25s}: None")

    print(f"\n  Aligning to ATLAS centrality grid...")
    pred = align_to_atlas(raw, ATLAS_CENT_CENTERS)

    print("\nCreating plot...")
    fig = make_plot(pred, atlas, cms, CORRECTION)

    out_path = Path(OUTPUT_FOLDER) / f"global_v0_{CORRECTION}_vs_exp_clean.pdf"
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: {out_path}")
    plt.close(fig)

    print("\n" + "="*70)
    print("✓ Done")
    print("="*70 + "\n")


EMULATOR PREDICTIONS vs EXPERIMENTAL DATA  —  Global v0
Correction: Grad  |  pT ref: 0.5–2 GeV/c

Loading ATLAS 5.02 TeV data...
  ✓ ATLAS 5.02 TeV  9 bins  cent=[ 2.5  7.5 15.  25.  35.  45.  55.  65.  75. ]

Loading CMS 2.76 TeV data (ROOT)...
  ✓ CMS 2.76 TeV  cent=[55. 65.]  v0=[0.01562 0.01901]  stat=[0.00059 0.00077]  syst=[7.e-05 9.e-05]

Loading emulator predictions...
  Keys in prediction dict: ['prior', 'original_posterior', 'posterior', 'observable_labels', 'centrality_centers']
  prior                    : (2000, 10)
  original_posterior       : (20000, 10)
  posterior                : (1400, 10)

  Aligning to ATLAS centrality grid...
    Stored emulator centres : [ 5. 15. 25. 35. 45. 55. 65. 75. 85. 95.]
    Target ATLAS centres    : [ 2.5  7.5 15.  25.  35.  45.  55.  65.  75. ]
    prior                    : (2000, 10) → (2000, 9)
    original_posterior       : (20000, 10) → (20000, 9)
    posterior                : (1400, 10) → (1400, 9)

Creating plot...
  ✓ Saved: p

# Model predictions vs experimental data

$v_0(p_T)$

In [11]:
#!/usr/bin/env python3
"""
Emulator predictions vs experimental data — v0(pT).

Posteriors:
  - Prior                      → grey hatch band
  - original_posterior         → orange dashed band  (pT-integrated)
  - posterior (Scaled Spectra) → blue solid band      (mcmc_chain_{corr}.csv)

Data:
  - ATLAS 5.02 TeV, 0.5–2 GeV  → open circle, golden     (CSV)
  - CMS   2.76 TeV, 0.5–2 GeV  → filled circle, golden   (ROOT parser)

ROOT KEY NOTES
--------------
  ATLAS cent7.csv  → 50-60%  (HEPData Figure_4a_cent7)
  ATLAS cent8.csv  → 60-70%  (HEPData Figure_4a_cent8)
  CMS key "5060"   → 50-60%
  CMS key "6070"   → 60-70%

  Emulator centrality index mapping (0-based):
    0 → 0-5%   1 → 5-10%   2 → 10-20%   3 → 20-30%   4 → 30-40%
    5 → 40-50%  6 → 50-60%  7 → 60-70%
"""

import struct
import zlib
import re
import math
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

# =============================================================================
# STYLE  (Gardim — identical to global v0 script)
# =============================================================================

plt.rcParams.update({
    "xtick.major.size":     6,    "ytick.major.size":     6,
    "xtick.minor.size":     3,    "ytick.minor.size":     3,
    "xtick.major.width":    1.2,  "ytick.major.width":    1.2,
    "xtick.minor.width":    1.0,  "ytick.minor.width":    1.0,
    "axes.linewidth":       1.2,
    "lines.linewidth":      1.5,
    "xtick.direction":      "in", "ytick.direction":      "in",
    "xtick.minor.visible":  True, "ytick.minor.visible":  True,
    "xtick.major.pad":      4,    "ytick.major.pad":      4,
    "legend.handletextpad": 0.4,
    "text.usetex":          False,
    "font.family":          "serif",
    "font.size":            14,
})

# =============================================================================
# CONFIGURATION  — edit these two lines to change centrality
# =============================================================================

# Centrality: use the string key that matches both emulator index and CMS ROOT key
# Emulator index 6 → 50-60%,  CMS key "5060",  ATLAS cent7
# Emulator index 7 → 60-70%,  CMS key "6070",  ATLAS cent8
CENTRALITY_INDEX = 6          # emulator centrality bin (0-based)
CMS_CENT_KEY     = "5060"     # matching ROOT object suffix

# ATLAS file: cent7 = 50-60%, cent8 = 60-70%
ATLAS_CENT_NUM   = 7          # number in ATLAS filename

EXP_V0PT_FILE = (
    "experimental_data/differential_radial_flow/"
    f"HEPData-ins2907010-v1-Figure_4a_cent{ATLAS_CENT_NUM}.csv"
)
CMS_MAIN_ROOT = "main.root"
CMS_SYST_ROOT = "SystUncs.root"

PCA_RESULTS_V0PT_FILE = "pca_results/pca_results_v0pt_all_corrections.pkl"
EMULATOR_FOLDER_V0PT  = "emulators_v0pt"   # 9 emulators for v0(pT)

RANGE_FILE              = "design_pts_Pb_Pb_2760_production/design_ranges_main_PbPb-2760.dat"
DESIGN_PTS_FILE         = "design_pts_Pb_Pb_2760_production/design_pts_main_PbPb-2760.dat"
ORIGINAL_POSTERIOR_FILE = "new_LHC_posterior_samples.csv"

CORRECTION    = "Grad"
OUTPUT_FOLDER = "predictions_v0pt_vs_exp"

N_PRIOR_SAMPLES = 2000
N_PC_USE_V0PT   = None   # None = use all available PCs

# Colours — same as global v0 script
ATLAS_COLOR = "#b8860b"    # dark golden    → open circle
CMS_COLOR   = "#b8860b"    # same golden family → filled circle
PRIOR_COLOR = "grey"
POST1_COLOR = "darkorange"
POST2_COLOR = "#1f6fbd"

CENT_EDGES  = [0, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90]
CENT_LABELS = [f"{CENT_EDGES[i]}-{CENT_EDGES[i+1]}%"
               for i in range(len(CENT_EDGES)-1)]

Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

# =============================================================================
# CMS ROOT PARSER  (v0(pT) differential)
# =============================================================================

def _decompress_root(filepath):
    raw    = Path(filepath).read_bytes()
    blocks = []
    i = 0
    while i < len(raw) - 2:
        if raw[i] == 0x78 and raw[i+1] in (0x9C, 0x01, 0xDA, 0x5E):
            try:
                blocks.append(zlib.decompress(raw[i:]))
            except zlib.error:
                pass
        i += 1
    return blocks


def _block_name(block):
    text = block.decode("latin-1")
    m = re.search(
        r"(sv0pt|v0ptv0|v0pt|gr_sv0pt|gr_v0ptv0|gr_v0pt|gr_v0)[_\w]*",
        text
    )
    return m.group(0) if m else ""


def _read_doubles(block, offset, n):
    return [struct.unpack(">d", block[offset + k*8: offset + k*8 + 8])[0]
            for k in range(n)]


def _find_x_offset(block, n=18):
    for i in range(70, 140):
        if i + n*8 > len(block):
            break
        trial = _read_doubles(block, i, n)
        if (all(0.3 < v < 15 for v in trial) and
                all(trial[k] < trial[k+1] for k in range(n-1))):
            return i
    return None


def _read_tgrapherrors(block, n=18):
    x_off = _find_x_offset(block, n)
    if x_off is None:
        return None
    y_off = x_off + n*8 + 1
    ey_off = y_off + n*8 + 57
    for ey_try in range(y_off + n*8, len(block) - n*8):
        trial = _read_doubles(block, ey_try, n)
        if all(0 < v < 5 for v in trial):
            ey_off = ey_try
            break
    return {
        "x":  np.array(_read_doubles(block, x_off,  n)),
        "y":  np.array(_read_doubles(block, y_off,   n)),
        "ey": np.array(_read_doubles(block, ey_off,  n)),
    }


def _read_tgraphasymerrors(block, n=18):
    x_off = _find_x_offset(block, n)
    if x_off is None:
        return None
    y_off   = x_off   + n*8 + 1
    exl_off = y_off   + n*8 + 57
    exh_off = exl_off + n*8 + 1
    eyl_off = exh_off + n*8 + 1
    eyh_off = eyl_off + n*8 + 1
    if eyh_off + n*8 > len(block):
        return None
    return {
        "x":   np.array(_read_doubles(block, x_off,   n)),
        "y":   np.array(_read_doubles(block, y_off,    n)),
        "exl": np.array(_read_doubles(block, exl_off,  n)),
        "exh": np.array(_read_doubles(block, exh_off,  n)),
        "eyl": np.array(_read_doubles(block, eyl_off,  n)),
        "eyh": np.array(_read_doubles(block, eyh_off,  n)),
    }


def load_cms_v0pt(main_path, syst_path, cent_key):
    mp, sp = Path(main_path), Path(syst_path)
    if not mp.exists() or not sp.exists():
        print(f"  WARNING: CMS ROOT files not found.")
        return None

    stat_key = f"v0pt_ptref_1_{cent_key}"
    syst_key = f"gr_v0pt_{cent_key}_total"

    stat_obj = None
    for b in _decompress_root(mp):
        if stat_key in _block_name(b):
            r = _read_tgrapherrors(b)
            if r is not None:
                stat_obj = r
                break

    if stat_obj is None:
        print(f"  WARNING: '{stat_key}' not found in main.root")
        return None

    syst_obj = None
    for b in _decompress_root(sp):
        name = _block_name(b)
        if syst_key in name and "_total" in name:
            r = _read_tgraphasymerrors(b)
            if r is not None:
                syst_obj = r
                break

    if syst_obj is None:
        print(f"  WARNING: '{syst_key}' not found in SystUncs.root — syst set to 0")
        syst_err = np.zeros_like(stat_obj["ey"])
        pT_exl = pT_exh = np.zeros_like(stat_obj["x"])
    else:
        syst_err = (syst_obj["eyl"] + syst_obj["eyh"]) / 2
        pT_exl   = syst_obj["exl"]
        pT_exh   = syst_obj["exh"]

    stat_err = stat_obj["ey"]
    mask     = stat_err > 1e-10

    pT    = stat_obj["x"][mask]
    v0pt  = stat_obj["y"][mask]
    stat  = stat_err[mask]
    syst  = syst_err[mask]
    total = np.sqrt(stat**2 + syst**2)

    print(f"  ✓ CMS 2.76 TeV v0(pT)  cent={cent_key}  "
          f"{len(pT)} points  pT=[{pT.min():.3f},{pT.max():.2f}] GeV/c")
    print(f"    v0pt range: [{v0pt.min():.4f}, {v0pt.max():.4f}]")
    print(f"    stat range: [{stat.min():.4f}, {stat.max():.4f}]")

    return {
        "pT":    pT,
        "pT_exl": pT_exl[mask] if syst_obj is not None else np.zeros_like(pT),
        "pT_exh": pT_exh[mask] if syst_obj is not None else np.zeros_like(pT),
        "v0pt":  v0pt,
        "stat":  stat,
        "syst":  syst,
        "total": total,
    }

# =============================================================================
# LOAD ATLAS DATA
# =============================================================================

def load_atlas_v0pt(filepath):
    if not Path(filepath).exists():
        print(f"  WARNING: ATLAS file not found: {filepath}")
        return None
    data = pd.read_csv(filepath, comment='#')
    data.columns = ['pT', 'v0', 'stat_plus', 'stat_minus',
                    'sys_plus', 'sys_minus']
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')
    stat  = np.abs(data['stat_plus'].values)
    syst  = np.abs(data['sys_plus'].values)
    total = np.sqrt(stat**2 + syst**2)
    print(f"  ✓ ATLAS 5.02 TeV v0(pT)  {len(data)} bins  "
          f"pT=[{data['pT'].values.min():.2f},{data['pT'].values.max():.2f}]")
    return {
        "pT":   data['pT'].values,
        "v0":   data['v0'].values,
        "stat": stat,
        "syst": syst,
        "total": total,
    }

# =============================================================================
# LOAD DESIGN RANGES + POSTERIOR SAMPLES
# =============================================================================

print("\n" + "="*70)
print("EMULATOR PREDICTIONS vs EXPERIMENTAL DATA  —  v0(pT)")
print(f"Correction: {CORRECTION}  |  Centrality: {CENT_LABELS[CENTRALITY_INDEX]}")
print("="*70)

design_range = pd.read_csv(RANGE_FILE, index_col=0)
param_min = design_range["min"].values
param_max = design_range["max"].values
n_params  = len(param_min)

orig_df      = pd.read_csv(ORIGINAL_POSTERIOR_FILE)
orig_samples = orig_df.iloc[:, :-1].values
print(f"  ✓ {n_params} parameters, {len(orig_samples)} original posterior samples")

# =============================================================================
# LOAD PCA + EMULATORS
# =============================================================================

with open(PCA_RESULTS_V0PT_FILE, "rb") as f:
    pca_results = pickle.load(f)

pca_info   = pca_results[CORRECTION]
obs_labels = pca_info["observable_labels"]

# N_TRAIN = number of design points used in PCA (after removing problematic ones)
# This is the fixed denominator for the inverse PCA transform.
N_TRAIN = pca_info['pc_scores'].shape[0]
print(f"  ✓ N_TRAIN (training design points): {N_TRAIN}")

# pT values and column indices for the requested centrality bin
pt_values = np.array([lbl[1] for lbl in obs_labels
                       if lbl[0] == CENTRALITY_INDEX])
cent_cols  = np.array([j for j, lbl in enumerate(obs_labels)
                        if lbl[0] == CENTRALITY_INDEX])

print(f"  ✓ PCA: {pca_info['n_components']} PCs, "
      f"{len(pt_values)} pT bins for cent {CENTRALITY_INDEX} "
      f"({CENT_LABELS[CENTRALITY_INDEX]}) "
      f"pT=[{pt_values.min():.3f}–{pt_values.max():.3f}] GeV/c")

emu_path = Path(EMULATOR_FOLDER_V0PT) / f"gp_emulators_v0pt_{CORRECTION}.pkl"
if not emu_path.exists():
    raise FileNotFoundError(f"Emulator not found: {emu_path}")

with open(emu_path, "rb") as f:
    emulators = pickle.load(f)

n_pc = min(len(emulators), pca_info['n_components'])
if N_PC_USE_V0PT is not None:
    n_pc = min(n_pc, N_PC_USE_V0PT)
emulators = emulators[:n_pc]
print(f"  ✓ Emulators loaded: {n_pc} PCs")

inv_tf = pca_info["inverse_tf_matrix"][:n_pc, :]
scaler = pca_info["scaler"]

# =============================================================================
# EMULATOR SANITY CHECK — verify emulators vary with input
# =============================================================================

print("\n--- Emulator sanity check ---")

# Load design points to test on training data
design_ok = False
if Path(DESIGN_PTS_FILE).exists():
    design_df = pd.read_csv(DESIGN_PTS_FILE, index_col=0)
    design_theta = design_df.values  # shape (N_design, 17)

    # Remove design points excluded from PCA training (if tracked)
    removed = pca_info.get('removed_design_points', [])
    if len(removed) > 0:
        mask_keep = np.ones(len(design_df), dtype=bool)
        # removed may be indices or design-point IDs
        dp_ids = pca_info.get('design_point_ids', None)
        if dp_ids is not None:
            removed_set = set(removed)
            for idx, dp_id in enumerate(design_df.index):
                if dp_id in removed_set:
                    mask_keep[idx] = False
        design_theta = design_theta[mask_keep]

    print(f"  Design points for test: {design_theta.shape[0]}")

    # Test first 5 design points
    test_theta = design_theta[:5]
    print(f"  PC score predictions vs true (first 5 design points, PC0):")
    pc0_preds = emulators[0].predict(test_theta)
    pc0_true  = pca_info['pc_scores'][:5, 0]
    for i in range(5):
        print(f"    dp{i}: pred={pc0_preds[i]:.4f}  true={pc0_true[i]:.4f}")

    # Test sensitivity: does the emulator vary with theta?
    print(f"\n  Random prior samples — PC0 predictions (should vary):")
    np.random.seed(42)
    test_rand = np.random.uniform(param_min, param_max, (5, n_params))
    pc0_rand = emulators[0].predict(test_rand)
    print(f"    {pc0_rand.flatten()}")
    print(f"    std = {pc0_rand.std():.6f}  (should be >> 0)")
    design_ok = True
else:
    print(f"  WARNING: design pts file not found ({DESIGN_PTS_FILE}), skipping check")

# Check reconstruction from true PC scores
print(f"\n  Reconstruction check (true pc_scores → Y_data):")
pc_true = pca_info['pc_scores'][:3]   # first 3 training points
inv_tf_full = pca_info["inverse_tf_matrix"][:n_pc, :]

# From the PCA notebook: reconstruct_from_pca uses / sqrt(n_samples - 1)
# where n_samples = pc_scores.shape[0] = N_TRAIN
Y_std_check  = (pc_true @ inv_tf_full) / math.sqrt(N_TRAIN - 1)
Y_recon      = scaler.inverse_transform(Y_std_check)
Y_true       = pca_info['Y_data'][:3]
err = np.abs(Y_recon - Y_true).max()
print(f"    Max reconstruction error (formula A: /sqrt(N_TRAIN-1)): {err:.6e}")

# Alternative: without the division
Y_std_b  = pc_true @ inv_tf_full
Y_recon_b = scaler.inverse_transform(Y_std_b)
err_b = np.abs(Y_recon_b - Y_true).max()
print(f"    Max reconstruction error (formula B: no division):       {err_b:.6e}")

# Which formula is correct?
if err < err_b:
    USE_SQRT_NTRAIN = True
    print(f"  ==> Formula A is correct (divide by sqrt(N_TRAIN-1) = sqrt({N_TRAIN-1}))")
else:
    USE_SQRT_NTRAIN = False
    print(f"  ==> Formula B is correct (no extra division)")

print("--- End sanity check ---\n")

# =============================================================================
# PREDICTION ENGINE — formula selected automatically by sanity check
# =============================================================================

def predict_v0pt_cent(theta_batch):
    """
    Predict v0(pT) for all samples in theta_batch.

    CORRECT FORMULA (confirmed by diagnostic):
      Y = pc_pred @ inv_tf + scaler.mean_

    inv_tf already has scaler.scale_ absorbed inside it, so
    scaler.inverse_transform must NOT be used (it would multiply
    by scale_ a second time, collapsing all bands to a single line).
    """
    pc_pred = np.column_stack([emu.predict(theta_batch) for emu in emulators])
    Y_full  = pc_pred @ inv_tf + scaler.mean_
    return Y_full[:, cent_cols]   # shape (n_samples, n_pt_bins)


def run_predictions(samples, label=""):
    print(f"    {label}: {samples.shape[0]} samples ...", end=" ", flush=True)
    result = predict_v0pt_cent(samples)
    print(f"done  shape={result.shape}  "
          f"range=[{result.min():.4f}, {result.max():.4f}]")
    return result


print("Generating predictions...")
np.random.seed(1)
prior_samples = np.random.uniform(param_min, param_max, (N_PRIOR_SAMPLES, n_params))

prior_pred = run_predictions(prior_samples, "Prior")
orig_pred  = run_predictions(orig_samples,  "Original posterior (pT-integrated)")

post_pred = None
post_path = Path(f"posterior/mcmc_chain_{CORRECTION}.csv")
if post_path.exists():
    post_samples = pd.read_csv(post_path).values
    post_pred    = run_predictions(post_samples, f"Scaled Spectra ({CORRECTION})")
else:
    print(f"    WARNING: {post_path} not found — blue band skipped")

for label, arr in [("prior", prior_pred), ("original", orig_pred),
                   ("scaled spectra", post_pred)]:
    if arr is not None:
        spread = arr.std(axis=0).mean()
        print(f"  Spread check — {label}: mean σ = {spread:.6f}")

# =============================================================================
# LOAD EXPERIMENTAL DATA
# =============================================================================

print("\nLoading experimental data...")
atlas = load_atlas_v0pt(EXP_V0PT_FILE)
cms   = load_cms_v0pt(CMS_MAIN_ROOT, CMS_SYST_ROOT, CMS_CENT_KEY)

# =============================================================================
# PLOT HELPERS
# =============================================================================

def configure_axis(ax, fs):
    ax.yaxis.set_ticks_position('both')
    ax.xaxis.set_ticks_position('both')
    ax.tick_params(which='both', labelsize=fs)


def fill_band(ax, x, arr, color, alpha, label=None, hatch=None):
    lo, hi = np.percentile(arr, [5, 95], axis=0)
    kw = dict(alpha=alpha, label=label, zorder=2)
    if hatch:
        kw.update(hatch=hatch, edgecolor=color, facecolor=color)
    else:
        kw.update(color=color)
    ax.fill_between(x, lo, hi, **kw)


def fill_band_dashed(ax, x, arr, color, alpha, label=None):
    lo, hi = np.percentile(arr, [5, 95], axis=0)
    ax.fill_between(x, lo, hi, color=color, alpha=alpha,
                    label=label, zorder=2)
    ax.plot(x, lo, color=color, lw=1.5, ls='--', alpha=0.9, zorder=3)
    ax.plot(x, hi, color=color, lw=1.5, ls='--', alpha=0.9, zorder=3)


def fill_ratio_band(ax, x_pred, arr, exp_v0, exp_pt, color, alpha):
    lo, hi = np.percentile(arr, [5, 95], axis=0)
    lo_i = np.interp(exp_pt, x_pred, lo)
    hi_i = np.interp(exp_pt, x_pred, hi)
    valid = np.abs(exp_v0) > 1e-5
    if valid.any():
        ax.fill_between(exp_pt[valid],
                        lo_i[valid] / exp_v0[valid],
                        hi_i[valid] / exp_v0[valid],
                        color=color, alpha=alpha, zorder=2)


def fill_ratio_band_dashed(ax, x_pred, arr, exp_v0, exp_pt, color, alpha):
    lo, hi = np.percentile(arr, [5, 95], axis=0)
    lo_i = np.interp(exp_pt, x_pred, lo)
    hi_i = np.interp(exp_pt, x_pred, hi)
    valid = np.abs(exp_v0) > 1e-5
    if valid.any():
        ax.fill_between(exp_pt[valid],
                        lo_i[valid] / exp_v0[valid],
                        hi_i[valid] / exp_v0[valid],
                        color=color, alpha=alpha, zorder=2)
        ax.plot(exp_pt[valid], lo_i[valid] / exp_v0[valid],
                color=color, lw=1.5, ls='--', alpha=0.9, zorder=3)
        ax.plot(exp_pt[valid], hi_i[valid] / exp_v0[valid],
                color=color, lw=1.5, ls='--', alpha=0.9, zorder=3)


def syst_box_log(ax, xc, yc, sy, exl, exh, color, lw=1.3):
    ax.add_patch(plt.Rectangle(
        (xc - exl, yc - sy), exl + exh, 2*sy,
        lw=lw, edgecolor=color, facecolor=color, alpha=0.25, zorder=4))

# =============================================================================
# MAIN PLOT
# =============================================================================

def make_plot(pt, prior, orig, post, atlas, cms, correction, cent_label):
    fs = 18

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(9, 10),
        gridspec_kw={"height_ratios": [3, 1]},
        sharex=True,
    )

    # ── 1. Prior — grey hatched band ─────────────────────────────────────────
    fill_band(ax_top, pt, prior,
              color=PRIOR_COLOR, alpha=0.30,
              label="Prior 90% C.I.", hatch="///")
    if atlas is not None:
        fill_ratio_band(ax_bot, pt, prior,
                        atlas['v0'], atlas['pT'],
                        color=PRIOR_COLOR, alpha=0.25)

    # ── 2. pT-integrated posterior — orange dashed border ────────────────────
    fill_band_dashed(ax_top, pt, orig,
                     color=POST1_COLOR, alpha=0.35,
                     label=r"Posterior 90% C.I. ($p_T$-integrated)")
    if atlas is not None:
        fill_ratio_band_dashed(ax_bot, pt, orig,
                               atlas['v0'], atlas['pT'],
                               color=POST1_COLOR, alpha=0.30)

    # ── 3. Scaled Spectra posterior — blue solid band ─────────────────────────
    if post is not None:
        fill_band(ax_top, pt, post,
                  color=POST2_COLOR, alpha=0.40,
                  label=r"Posterior 90% C.I. (Scaled Spectra)")
        if atlas is not None:
            fill_ratio_band(ax_bot, pt, post,
                            atlas['v0'], atlas['pT'],
                            color=POST2_COLOR, alpha=0.35)

    # ── 4. ATLAS data — open circle, golden ───────────────────────────────────
    if atlas is not None:
        ax_top.errorbar(
            atlas['pT'], atlas['v0'],
            yerr=atlas['stat'],
            fmt='s', mfc='none', color=ATLAS_COLOR,
            markersize=8, capsize=4, lw=1.8, zorder=10,
            label=r"ATLAS 5.02 TeV  ($0.5\!-\!2$ GeV/$c$)",
        )
        for xc, yc, sy in zip(atlas['pT'], atlas['v0'], atlas['syst']):
            bw = xc * 0.07
            ax_top.add_patch(plt.Rectangle(
                (xc-bw, yc-sy), 2*bw, 2*sy,
                lw=1.3, edgecolor=ATLAS_COLOR, facecolor='none', zorder=11))

        valid = np.abs(atlas['v0']) > 1e-5
        ax_bot.errorbar(
            atlas['pT'][valid],
            np.ones(valid.sum()),
            yerr=atlas['total'][valid] / atlas['v0'][valid],
            fmt='s', mfc='none', color=ATLAS_COLOR,
            markersize=8, capsize=4, lw=1.8, zorder=10,
        )
        for xc, sy, yv in zip(atlas['pT'][valid],
                               atlas['syst'][valid],
                               atlas['v0'][valid]):
            bw = xc * 0.07
            ax_bot.add_patch(plt.Rectangle(
                (xc-bw, 1.0-sy/yv), 2*bw, 2*sy/yv,
                lw=1.3, edgecolor=ATLAS_COLOR, facecolor='none', zorder=11))

    # ── 5. CMS data — filled circle, same golden family ───────────────────────
    if cms is not None:
        cms_mask = cms['pT'] <= pt.max() * 1.1
        ax_top.errorbar(
            cms['pT'][cms_mask], cms['v0pt'][cms_mask],
            yerr=cms['stat'][cms_mask],
            fmt='s', mfc=CMS_COLOR, color=CMS_COLOR,
            markersize=8, capsize=4, lw=1.8, zorder=10,
            label=r"CMS 2.76 TeV  ($0.5\!-\!2$ GeV/$c$)",
        )
        for xc, yc, sy, exl, exh in zip(
                cms['pT'][cms_mask], cms['v0pt'][cms_mask],
                cms['syst'][cms_mask],
                cms['pT_exl'][cms_mask], cms['pT_exh'][cms_mask]):
            syst_box_log(ax_top, xc, yc, sy, exl, exh, CMS_COLOR)

        if atlas is not None:
            atlas_at_cms = np.interp(cms['pT'][cms_mask],
                                     atlas['pT'], atlas['v0'])
            valid = np.abs(atlas_at_cms) > 1e-6
            if valid.any():
                atlas_err_cms = np.interp(
                    cms['pT'][cms_mask][valid],
                    atlas['pT'], atlas['total']
                )
                ratio_cms = cms['v0pt'][cms_mask][valid] / atlas_at_cms[valid]
                rerr_cms  = np.sqrt(
                    (cms['stat'][cms_mask][valid] / cms['v0pt'][cms_mask][valid])**2 +
                    (atlas_err_cms / atlas_at_cms[valid])**2
                ) * np.abs(ratio_cms)
                ax_bot.errorbar(
                    cms['pT'][cms_mask][valid], ratio_cms,
                    yerr=rerr_cms,
                    fmt='s', mfc=CMS_COLOR, color=CMS_COLOR,
                    markersize=8, capsize=4, lw=1.8, zorder=10,
                    label=r"CMS$\,/\,$ATLAS",
                )

    # ── Formatting ────────────────────────────────────────────────────────────
    ax_top.set_xscale('log')
    ax_top.set_xlim(pt.min()*0.80, pt.max()*1.20)
    ax_top.set_ylabel(r"$v_0(p_T)$", fontsize=fs+8)
    ax_top.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
    ax_top.legend(loc='upper left', frameon=False, fontsize=fs-5)
    ax_top.text(0.97, 0.03,
                f"Pb+Pb,  {cent_label}\n"
                r"$p_T^{\rm ref} = 0.5\!-\!2$ GeV/$c$",
                transform=ax_top.transAxes,
                va='bottom', ha='right', fontsize=fs-4)
    configure_axis(ax_top, fs-2)

    ax_bot.axhline(1.0, color='k', ls='--', lw=1.5)
    ax_bot.set_xscale('log')
    ax_bot.set_xlim(pt.min()*0.80, pt.max()*1.20)
    ax_bot.set_ylim(-3.0, 5.0)
    ax_bot.set_xlabel(r"$p_T$ [GeV/$c$]", fontsize=fs+4)
    ax_bot.set_ylabel("Model / ATLAS", fontsize=fs+2)
    ax_bot.grid(True, alpha=0.25, which='major')
    ax_bot.axhline(0, color='k', ls=':', lw=0.8, alpha=0.4)
    handles, labels = ax_bot.get_legend_handles_labels()
    if handles:
        ax_bot.legend(handles, labels, fontsize=fs-6,
                      loc='upper right', frameon=False)
    configure_axis(ax_bot, fs-2)

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.05)
    return fig

# =============================================================================
# MAIN
# =============================================================================

print("\nCreating plot...")
fig = make_plot(
    pt         = pt_values,
    prior      = prior_pred,
    orig       = orig_pred,
    post       = post_pred,
    atlas      = atlas,
    cms        = cms,
    correction = CORRECTION,
    cent_label = CENT_LABELS[CENTRALITY_INDEX],
)

label_str = CENT_LABELS[CENTRALITY_INDEX].replace('-','').replace('%','')
out_path  = Path(OUTPUT_FOLDER) / f"v0pt_{CORRECTION}_{label_str}_vs_exp.pdf"
fig.savefig(out_path, dpi=300, bbox_inches='tight')
print(f"  ✓ Saved: {out_path}")
plt.close(fig)

print("\n" + "="*70)
print("✓ Done")
print("="*70 + "\n")


EMULATOR PREDICTIONS vs EXPERIMENTAL DATA  —  v0(pT)
Correction: Grad  |  Centrality: 50-60%
  ✓ 17 parameters, 20000 original posterior samples
  ✓ N_TRAIN (training design points): 485
  ✓ PCA: 9 PCs, 13 pT bins for cent 6 (50-60%) pT=[0.552–2.494] GeV/c
  ✓ Emulators loaded: 9 PCs

--- Emulator sanity check ---

  Reconstruction check (true pc_scores → Y_data):
    Max reconstruction error (formula A: /sqrt(N_TRAIN-1)): 3.961783e-02
    Max reconstruction error (formula B: no division):       3.805257e-02
  ==> Formula B is correct (no extra division)
--- End sanity check ---

Generating predictions...
    Prior: 2000 samples ... done  shape=(2000, 13)  range=[-0.1352, 0.3621]
    Original posterior (pT-integrated): 20000 samples ... done  shape=(20000, 13)  range=[-0.0610, 0.2788]
    Scaled Spectra (Grad): 1400 samples ... done  shape=(1400, 13)  range=[-0.1516, 0.3345]
  Spread check — prior: mean σ = 0.023774
  Spread check — original: mean σ = 0.007524
  Spread check — scaled 

# Model predictions vs experimental data

$v_0(p_T)/v_0$

In [13]:
#!/usr/bin/env python3
"""
Emulator predictions vs experimental data — v0(pT) / v0 ratio.

For each sample the ratio is computed point-by-point:
  ratio(pT) = v0(pT) / v0_global

Both v0(pT) and v0_global are predicted by their respective emulators
using the SAME parameter sample, so the ratio is fully correlated.

Posteriors:
  - Prior                      → grey hatch band
  - original_posterior         → orange dashed band  (pT-integrated)
  - posterior (Scaled Spectra) → blue solid band      (mcmc_chain_{corr}.csv)

Data:
  - ATLAS 5.02 TeV  → open circle, golden
  - CMS   2.76 TeV  → filled circle, golden
"""

import struct
import zlib
import re
import math
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

# =============================================================================
# STYLE
# =============================================================================

plt.rcParams.update({
    "xtick.major.size":     6,    "ytick.major.size":     6,
    "xtick.minor.size":     3,    "ytick.minor.size":     3,
    "xtick.major.width":    1.2,  "ytick.major.width":    1.2,
    "xtick.minor.width":    1.0,  "ytick.minor.width":    1.0,
    "axes.linewidth":       1.2,
    "lines.linewidth":      1.5,
    "xtick.direction":      "in", "ytick.direction":      "in",
    "xtick.minor.visible":  True, "ytick.minor.visible":  True,
    "xtick.major.pad":      4,    "ytick.major.pad":      4,
    "legend.handletextpad": 0.4,
    "text.usetex":          False,
    "font.family":          "serif",
    "font.size":            14,
})

# =============================================================================
# CONFIGURATION
# =============================================================================

CENTRALITY_INDEX = 6          # emulator centrality bin (0-based) = 50-60%
CMS_CENT_KEY     = "5060"
ATLAS_CENT_NUM   = 7          # cent7.csv = 50-60%

# v0(pT) emulator
PCA_V0PT_FILE    = "pca_results/pca_results_v0pt_all_corrections.pkl"
EMU_V0PT_FOLDER  = "emulators_v0pt"

# v0 global emulator (3 PCs)
PCA_V0_FILE      = "pca_results/pca_results_global_v0_all_corrections.pkl"
EMU_V0_FOLDER    = "emulators_global_v0"
V0_CENT_INDEX    = CENTRALITY_INDEX   # same bin index

# Experimental data
ATLAS_V0PT_FILE = (
    "experimental_data/differential_radial_flow/"
    f"HEPData-ins2907010-v1-Figure_4a_cent{ATLAS_CENT_NUM}.csv"
)
ATLAS_V0_FILE   = "experimental_data/differential_radial_flow/HEPData-ins2907010-v1-Figure_2a.csv"
CMS_MAIN_ROOT   = "main.root"
CMS_SYST_ROOT   = "SystUncs.root"

RANGE_FILE              = "design_pts_Pb_Pb_2760_production/design_ranges_main_PbPb-2760.dat"
ORIGINAL_POSTERIOR_FILE = "new_LHC_posterior_samples.csv"

CORRECTION    = "Grad"
OUTPUT_FOLDER = "predictions_v0pt_vs_exp"

N_PRIOR_SAMPLES  = 2000
N_PC_USE_V0PT    = None   # None = all available
N_PC_USE_V0      = 3      # 3 PCs for global v0

CENT_EDGES  = [0, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90]
CENT_LABELS = [f"{CENT_EDGES[i]}-{CENT_EDGES[i+1]}%"
               for i in range(len(CENT_EDGES)-1)]

ATLAS_COLOR = "#b8860b"
CMS_COLOR   = "#b8860b"
PRIOR_COLOR = "grey"
POST1_COLOR = "darkorange"
POST2_COLOR = "#1f6fbd"

Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

# =============================================================================
# CMS ROOT PARSER  (v0(pT) differential)
# =============================================================================

def _decompress_root(filepath):
    raw    = Path(filepath).read_bytes()
    blocks = []
    i = 0
    while i < len(raw) - 2:
        if raw[i] == 0x78 and raw[i+1] in (0x9C, 0x01, 0xDA, 0x5E):
            try:
                blocks.append(zlib.decompress(raw[i:]))
            except zlib.error:
                pass
        i += 1
    return blocks


def _block_name(block):
    text = block.decode("latin-1")
    m = re.search(
        r"(sv0pt|v0ptv0|v0pt|gr_sv0pt|gr_v0ptv0|gr_v0pt|gr_sv0|gr_v0|sv0)[_\w]*",
        text
    )
    return m.group(0) if m else ""


def _read_doubles(block, offset, n):
    return [struct.unpack(">d", block[offset + k*8: offset + k*8 + 8])[0]
            for k in range(n)]


def _find_x_offset(block, n=18):
    for i in range(70, 140):
        if i + n*8 > len(block):
            break
        trial = _read_doubles(block, i, n)
        if n == 1:
            # single-point graph (global v0): x is a pT-ref value
            if 0.3 < trial[0] < 15:
                return i
        else:
            if (all(0.3 < v < 15 for v in trial) and
                    all(trial[k] < trial[k+1] for k in range(n-1))):
                return i
    return None


def _read_tgrapherrors(block, n=18):
    x_off = _find_x_offset(block, n)
    if x_off is None:
        return None
    y_off  = x_off + n*8 + 1
    ey_off = y_off + n*8 + 57
    # Search for ey: small positive values (errors are always positive)
    # For v0(pT): 0 < ey < 5; for global v0: 0 < ey < 1
    for ey_try in range(y_off + n*8, len(block) - n*8):
        trial = _read_doubles(block, ey_try, n)
        if all(0 < v < 5 for v in trial):
            ey_off = ey_try
            break
    return {
        "x":  np.array(_read_doubles(block, x_off,  n)),
        "y":  np.array(_read_doubles(block, y_off,   n)),
        "ey": np.array(_read_doubles(block, ey_off,  n)),
    }


def _read_tgraphasymerrors(block, n=18):
    x_off = _find_x_offset(block, n)
    if x_off is None:
        return None
    y_off   = x_off   + n*8 + 1
    exl_off = y_off   + n*8 + 57
    exh_off = exl_off + n*8 + 1
    eyl_off = exh_off + n*8 + 1
    eyh_off = eyl_off + n*8 + 1
    if eyh_off + n*8 > len(block):
        return None
    return {
        "x":   np.array(_read_doubles(block, x_off,   n)),
        "y":   np.array(_read_doubles(block, y_off,    n)),
        "exl": np.array(_read_doubles(block, exl_off,  n)),
        "exh": np.array(_read_doubles(block, exh_off,  n)),
        "eyl": np.array(_read_doubles(block, eyl_off,  n)),
        "eyh": np.array(_read_doubles(block, eyh_off,  n)),
    }


def load_cms_v0pt(main_path, syst_path, cent_key):
    mp, sp = Path(main_path), Path(syst_path)
    if not mp.exists() or not sp.exists():
        print(f"  WARNING: CMS ROOT files not found.")
        return None

    stat_key = f"v0pt_ptref_1_{cent_key}"
    syst_key = f"gr_v0pt_{cent_key}_total"

    stat_obj = None
    for b in _decompress_root(mp):
        if stat_key in _block_name(b):
            r = _read_tgrapherrors(b)
            if r is not None:
                stat_obj = r
                break

    if stat_obj is None:
        print(f"  WARNING: '{stat_key}' not found in main.root")
        return None

    syst_obj = None
    for b in _decompress_root(sp):
        name = _block_name(b)
        if syst_key in name and "_total" in name:
            r = _read_tgraphasymerrors(b)
            if r is not None:
                syst_obj = r
                break

    if syst_obj is None:
        syst_err = np.zeros_like(stat_obj["ey"])
        pT_exl = pT_exh = np.zeros_like(stat_obj["x"])
    else:
        syst_err = (syst_obj["eyl"] + syst_obj["eyh"]) / 2
        pT_exl   = syst_obj["exl"]
        pT_exh   = syst_obj["exh"]

    stat_err = stat_obj["ey"]
    mask     = stat_err > 1e-10

    pT    = stat_obj["x"][mask]
    v0pt  = stat_obj["y"][mask]
    stat  = stat_err[mask]
    syst  = syst_err[mask]
    total = np.sqrt(stat**2 + syst**2)

    print(f"  ✓ CMS 2.76 TeV v0(pT)  cent={cent_key}  "
          f"{len(pT)} points  pT=[{pT.min():.3f},{pT.max():.2f}] GeV/c")

    return {
        "pT":     pT,
        "pT_exl": pT_exl[mask] if syst_obj is not None else np.zeros_like(pT),
        "pT_exh": pT_exh[mask] if syst_obj is not None else np.zeros_like(pT),
        "v0pt":   v0pt,
        "stat":   stat,
        "syst":   syst,
        "total":  total,
    }

# =============================================================================
# LOAD ATLAS v0(pT) differential
# =============================================================================

def load_atlas_v0pt(filepath):
    if not Path(filepath).exists():
        print(f"  WARNING: ATLAS v0pt file not found: {filepath}")
        return None
    data = pd.read_csv(filepath, comment='#')
    data.columns = ['pT', 'v0', 'stat_plus', 'stat_minus',
                    'sys_plus', 'sys_minus']
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')
    stat  = np.abs(data['stat_plus'].values)
    syst  = np.abs(data['sys_plus'].values)
    total = np.sqrt(stat**2 + syst**2)
    print(f"  ✓ ATLAS 5.02 TeV v0(pT)  {len(data)} bins  "
          f"pT=[{data['pT'].values.min():.2f},{data['pT'].values.max():.2f}]")
    return {
        "pT":   data['pT'].values,
        "v0":   data['v0'].values,
        "stat": stat, "syst": syst, "total": total,
    }

# =============================================================================
# LOAD ATLAS v0 global (single value per centrality)
# =============================================================================

def load_atlas_v0_global(filepath, cent_index):
    """Return the single v0 value for a given centrality bin."""
    if not Path(filepath).exists():
        print(f"  WARNING: ATLAS v0 global file not found: {filepath}")
        return None
    data = pd.read_csv(filepath, comment='#')
    for i, row in data.iterrows():
        if i > 0 and str(row.iloc[0]) == 'Centrality [%]':
            data = data.iloc[:i].copy(); break
    data.columns = ['centrality', 'v0', 'stat_plus', 'stat_minus',
                    'sys_plus',   'sys_minus']
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')
    stat  = np.abs(data['stat_plus'].values)
    syst  = np.abs(data['sys_plus'].values)
    total = np.sqrt(stat**2 + syst**2)
    print(f"  ✓ ATLAS v0 global: {len(data)} centrality bins")
    # cent_index 6 → 50-60% → ATLAS row index 6
    if cent_index >= len(data):
        print(f"  WARNING: cent_index={cent_index} out of range")
        return None
    return {
        "v0":    data['v0'].values[cent_index],
        "stat":  stat[cent_index],
        "syst":  syst[cent_index],
        "total": total[cent_index],
        "cent":  data['centrality'].values[cent_index],
    }

# =============================================================================
# LOAD CMS v0 global (single value for 50-60% or 60-70%)
# =============================================================================

def load_cms_v0_global(main_path, syst_path, cent_key):
    """
    Parse CMS global v0 (single value) from ROOT for one centrality.

    CMS ROOT key in main.root: sv0_ptref_1_{cent_key}  (TGraphErrors, 1 point)
    Syst key in SystUncs.root: gr_sv0_{cent_key}_total  (TGraphAsymmErrors, 1 point)

    Returns dict with scalar v0, stat, syst.
    """
    mp, sp = Path(main_path), Path(syst_path)
    if not mp.exists():
        print(f"  WARNING: {main_path} not found")
        return None

    stat_key = f"sv0_ptref_1_{cent_key}"
    syst_key = f"gr_sv0_{cent_key}_total"

    # ---- stat: TGraphErrors with n=1 point ----
    stat_obj = None
    for b in _decompress_root(mp):
        name = _block_name(b)
        if stat_key in name:
            # n=1 point: x is a single pT-ref value (~0.5-2 GeV), y is v0 scalar
            r = _read_tgrapherrors(b, n=1)
            if r is not None:
                stat_obj = r
                break

    if stat_obj is None:
        print(f"  WARNING: CMS global v0 key '{stat_key}' not found — "
              f"will estimate from v0pt mean")
        return None

    v0_val  = float(stat_obj["y"][0])
    stat_v0 = float(stat_obj["ey"][0])

    # ---- syst ----
    syst_v0 = 0.0
    if sp.exists():
        for b in _decompress_root(sp):
            name = _block_name(b)
            if syst_key in name and "total" in name:
                r = _read_tgraphasymerrors(b, n=1)
                if r is not None:
                    syst_v0 = float((r["eyl"][0] + r["eyh"][0]) / 2)
                    break

    print(f"  ✓ CMS global v0  cent={cent_key}  "
          f"v0={v0_val:.5f} ± {stat_v0:.5f} (stat) ± {syst_v0:.5f} (syst)")
    return {
        "v0":   v0_val,
        "stat": stat_v0,
        "syst": syst_v0,
        "total": math.sqrt(stat_v0**2 + syst_v0**2),
    }


def load_cms_ratio(main_path, syst_path, cent_key, v0glob_external=None):
    """
    Compute CMS v0(pT)/v0 ratio.

    Strategy (in priority order):
      1. Try to read CMS v0_global from ROOT (key sv0_ptref_1_{cent_key}).
      2. If not found, use v0glob_external (e.g. ATLAS v0_global for the
         same centrality — physically they are the same observable).
      3. If neither available, return None.

    The old approach of reading the pre-calculated ratio from ROOT is
    unreliable: the binary parser is calibrated for pT-axis values
    (0.3–15 GeV) and can confuse the y-axis of the ratio (~-3 to +13)
    with the x-axis, returning scrambled data.
    The fallback of averaging v0pt over pT_ref is WRONG — v0_global is
    the spectrum-weighted mean, not the arithmetic mean of v0pt bins.
    """
    # Load differential v0(pT)
    v0pt_data = load_cms_v0pt(main_path, syst_path, cent_key)
    if v0pt_data is None:
        return None

    # Try ROOT first, then fall back to external (ATLAS) v0_global
    v0glob = load_cms_v0_global(main_path, syst_path, cent_key)

    if v0glob is None:
        if v0glob_external is not None:
            v0glob = v0glob_external
            print(f"  ⚠ CMS v0_global not in ROOT — using external reference "
                  f"(ATLAS): {v0glob['v0']:.5f}")
        else:
            print("  WARNING: no CMS v0_global available — CMS ratio skipped")
            return None

    v0g = v0glob["v0"]
    if abs(v0g) < 1e-6:
        print("  WARNING: CMS global v0 ≈ 0, cannot compute ratio")
        return None

    # Ratio and error propagation (identical to compute_exp_ratio logic)
    yp   = v0pt_data["v0pt"]
    ratio = yp / v0g

    # stat: quadrature of relative errors; protect against v0pt ≈ 0
    with np.errstate(invalid='ignore', divide='ignore'):
        rel_pt = np.where(np.abs(yp) > 1e-8,
                          v0pt_data["stat"] / np.abs(yp), 0.0)
    rel_g  = abs(v0glob["stat"] / v0g) if abs(v0g) > 1e-8 else 0.0
    stat   = np.abs(ratio) * np.sqrt(rel_pt**2 + rel_g**2)

    with np.errstate(invalid='ignore', divide='ignore'):
        rel_pt_s = np.where(np.abs(yp) > 1e-8,
                            v0pt_data["syst"] / np.abs(yp), 0.0)
    rel_g_s  = abs(v0glob.get("syst", 0.0) / v0g) if abs(v0g) > 1e-8 else 0.0
    syst     = np.abs(ratio) * np.sqrt(rel_pt_s**2 + rel_g_s**2)

    total = np.sqrt(stat**2 + syst**2)

    print(f"  ✓ CMS ratio v0(pT)/v0  cent={cent_key}  "
          f"{len(ratio)} points  v0_global={v0g:.5f}  "
          f"ratio=[{ratio.min():.3f}, {ratio.max():.3f}]")

    return {
        "pT":   v0pt_data["pT"],
        "v0":   ratio,
        "stat": stat,
        "syst": syst,
        "total": total,
    }

# =============================================================================
# LOAD EMULATORS + PCA
# =============================================================================

print("\n" + "="*70)
print("EMULATOR PREDICTIONS vs EXPERIMENTAL DATA  —  v0(pT)/v0")
print(f"Correction: {CORRECTION}  |  Centrality: {CENT_LABELS[CENTRALITY_INDEX]}")
print("="*70)

design_range = pd.read_csv(RANGE_FILE, index_col=0)
param_min = design_range["min"].values
param_max = design_range["max"].values
n_params  = len(param_min)

orig_df      = pd.read_csv(ORIGINAL_POSTERIOR_FILE)
orig_samples = orig_df.iloc[:, :-1].values
print(f"  ✓ {n_params} parameters, {len(orig_samples)} original posterior samples")

# --- v0(pT) emulator ---
with open(PCA_V0PT_FILE, "rb") as f:
    pca_v0pt = pickle.load(f)[CORRECTION]

obs_labels_pt = pca_v0pt["observable_labels"]
pt_values  = np.array([lbl[1] for lbl in obs_labels_pt if lbl[0] == CENTRALITY_INDEX])
cent_cols  = np.array([j for j, lbl in enumerate(obs_labels_pt) if lbl[0] == CENTRALITY_INDEX])

emu_v0pt_path = Path(EMU_V0PT_FOLDER) / f"gp_emulators_v0pt_{CORRECTION}.pkl"
with open(emu_v0pt_path, "rb") as f:
    emus_v0pt = pickle.load(f)

n_pc_v0pt = min(len(emus_v0pt), pca_v0pt['n_components'])
if N_PC_USE_V0PT is not None:
    n_pc_v0pt = min(n_pc_v0pt, N_PC_USE_V0PT)
emus_v0pt = emus_v0pt[:n_pc_v0pt]
inv_v0pt  = pca_v0pt["inverse_tf_matrix"][:n_pc_v0pt, :]
mean_v0pt = pca_v0pt["scaler"].mean_
print(f"  ✓ v0(pT) emulators: {n_pc_v0pt} PCs, {len(pt_values)} pT bins, "
      f"cent {CENT_LABELS[CENTRALITY_INDEX]}")

# --- v0 global emulator (3 PCs) ---
with open(PCA_V0_FILE, "rb") as f:
    pca_v0 = pickle.load(f)[CORRECTION]

obs_labels_v0 = pca_v0["observable_labels"]

# Global v0 labels are strings like '0-10%', '10-20%', ..., '50-60%', ...
# Build the target string from CENT_EDGES using V0_CENT_INDEX.
target_label = CENT_LABELS[V0_CENT_INDEX]   # e.g. '50-60%'
v0_col = None
for j, lbl in enumerate(obs_labels_v0):
    if str(lbl).replace(' ', '') == target_label.replace(' ', ''):
        v0_col = j
        break
if v0_col is None:
    raise ValueError(
        f"Could not find '{target_label}' in global v0 obs_labels: {obs_labels_v0}"
    )
print(f"  ✓ v0 global obs_labels: {obs_labels_v0}  →  '{target_label}' = col {v0_col}")

emu_v0_path = Path(EMU_V0_FOLDER) / f"gp_emulators_global_v0_{CORRECTION}.pkl"
with open(emu_v0_path, "rb") as f:
    emus_v0 = pickle.load(f)

n_pc_v0 = min(len(emus_v0), pca_v0['n_components'], N_PC_USE_V0)
emus_v0 = emus_v0[:n_pc_v0]
inv_v0   = pca_v0["inverse_tf_matrix"][:n_pc_v0, :]
mean_v0  = pca_v0["scaler"].mean_
print(f"  ✓ v0 global emulators: {n_pc_v0} PCs")

# =============================================================================
# PREDICTION ENGINE
# =============================================================================

def predict_v0pt(theta_batch):
    """Returns v0(pT) for the requested centrality bin. Shape: (N, n_pt_bins)."""
    pc = np.column_stack([e.predict(theta_batch) for e in emus_v0pt])
    Y  = pc @ inv_v0pt + mean_v0pt
    return Y[:, cent_cols]


def predict_v0_global(theta_batch):
    """Returns v0 global for the requested centrality bin. Shape: (N,)."""
    pc = np.column_stack([e.predict(theta_batch) for e in emus_v0])
    Y  = pc @ inv_v0 + mean_v0
    return Y[:, v0_col]   # single value per sample


def predict_ratio(theta_batch):
    """
    Compute v0(pT)/v0 for each sample.
    Returns shape (N, n_pt_bins). Samples where v0≈0 are masked out.
    """
    v0pt   = predict_v0pt(theta_batch)         # (N, n_pt)
    v0glob = predict_v0_global(theta_batch)    # (N,)

    # avoid divide-by-zero
    valid  = np.abs(v0glob) > 1e-6
    ratio  = np.full_like(v0pt, np.nan)
    ratio[valid] = v0pt[valid] / v0glob[valid, np.newaxis]
    return ratio


def run_predictions(samples, label=""):
    print(f"    {label}: {samples.shape[0]} samples ...", end=" ", flush=True)
    result = predict_ratio(samples)
    n_valid = np.isfinite(result[:, 0]).sum()
    print(f"done  shape={result.shape}  valid={n_valid}  "
          f"range=[{np.nanmin(result):.3f}, {np.nanmax(result):.3f}]")
    return result


print("\nGenerating predictions...")
np.random.seed(1)
prior_samples = np.random.uniform(param_min, param_max, (N_PRIOR_SAMPLES, n_params))

prior_pred = run_predictions(prior_samples, "Prior")
orig_pred  = run_predictions(orig_samples,  "Original posterior (pT-integrated)")

post_pred = None
post_path = Path(f"posterior/mcmc_chain_{CORRECTION}.csv")
if post_path.exists():
    post_samples = pd.read_csv(post_path).values
    post_pred    = run_predictions(post_samples, f"Scaled Spectra ({CORRECTION})")
else:
    print(f"    WARNING: {post_path} not found — blue band skipped")

for label, arr in [("prior", prior_pred), ("original", orig_pred),
                   ("scaled spectra", post_pred)]:
    if arr is not None:
        spread = np.nanstd(arr, axis=0).mean()
        print(f"  Spread check — {label}: mean σ = {spread:.6f}")

# =============================================================================
# LOAD EXPERIMENTAL DATA
# =============================================================================

print("\nLoading experimental data...")
atlas_v0pt   = load_atlas_v0pt(ATLAS_V0PT_FILE)
atlas_v0glob = load_atlas_v0_global(ATLAS_V0_FILE, CENTRALITY_INDEX)
cms_v0pt     = load_cms_v0pt(CMS_MAIN_ROOT, CMS_SYST_ROOT, CMS_CENT_KEY)

# Compute experimental ratio
def compute_exp_ratio(v0pt_data, v0glob_data):
    """Divide the differential v0pt data by the global v0 scalar."""
    if v0pt_data is None or v0glob_data is None:
        return None
    v0g   = v0glob_data["v0"]
    if abs(v0g) < 1e-6:
        return None
    ratio = v0pt_data["v0"] / v0g
    # error propagation: σ(ratio)/ratio = sqrt((σ_v0pt/v0pt)² + (σ_v0/v0)²)
    rel_pt = v0pt_data["stat"] / np.abs(v0pt_data["v0"])
    rel_g  = v0glob_data["stat"] / abs(v0g)
    stat   = np.abs(ratio) * np.sqrt(rel_pt**2 + rel_g**2)

    rel_pt_s = v0pt_data["syst"] / np.abs(v0pt_data["v0"])
    rel_g_s  = v0glob_data["syst"] / abs(v0g)
    syst     = np.abs(ratio) * np.sqrt(rel_pt_s**2 + rel_g_s**2)

    total = np.sqrt(stat**2 + syst**2)
    return {
        "pT":   v0pt_data["pT"],
        "v0":   ratio,
        "stat": stat,
        "syst": syst,
        "total": total,
    }

atlas_ratio = compute_exp_ratio(atlas_v0pt, atlas_v0glob)
if atlas_ratio is not None:
    print(f"  ✓ ATLAS ratio v0(pT)/v0  range=[{atlas_ratio['v0'].min():.3f}, "
          f"{atlas_ratio['v0'].max():.3f}]")

# CMS ratio: try CMS ROOT v0_global first; fall back to ATLAS v0_global
# (same physical observable — CMS 2.76 TeV and ATLAS 5.02 TeV give compatible
#  v0_global values for the same centrality class)
cms_ratio = load_cms_ratio(CMS_MAIN_ROOT, CMS_SYST_ROOT, CMS_CENT_KEY,
                           v0glob_external=atlas_v0glob)
if cms_ratio is not None:
    print(f"  ✓ CMS ratio range=[{cms_ratio['v0'].min():.3f}, "
          f"{cms_ratio['v0'].max():.3f}]")

# =============================================================================
# PLOT HELPERS
# =============================================================================

def configure_axis(ax, fs):
    ax.yaxis.set_ticks_position('both')
    ax.xaxis.set_ticks_position('both')
    ax.tick_params(which='both', labelsize=fs)


def fill_band(ax, x, arr, color, alpha, label=None, hatch=None):
    lo = np.nanpercentile(arr, 5,  axis=0)
    hi = np.nanpercentile(arr, 95, axis=0)
    kw = dict(alpha=alpha, label=label, zorder=2)
    if hatch:
        kw.update(hatch=hatch, edgecolor=color, facecolor=color)
    else:
        kw.update(color=color)
    ax.fill_between(x, lo, hi, **kw)


def fill_band_dashed(ax, x, arr, color, alpha, label=None):
    lo = np.nanpercentile(arr, 5,  axis=0)
    hi = np.nanpercentile(arr, 95, axis=0)
    ax.fill_between(x, lo, hi, color=color, alpha=alpha,
                    label=label, zorder=2)
    ax.plot(x, lo, color=color, lw=1.5, ls='--', alpha=0.9, zorder=3)
    ax.plot(x, hi, color=color, lw=1.5, ls='--', alpha=0.9, zorder=3)


def fill_ratio_band(ax, x_pred, arr, exp_v0, exp_pt, color, alpha):
    lo = np.nanpercentile(arr, 5,  axis=0)
    hi = np.nanpercentile(arr, 95, axis=0)
    lo_i = np.interp(exp_pt, x_pred, lo)
    hi_i = np.interp(exp_pt, x_pred, hi)
    valid = np.abs(exp_v0) > 1e-5
    if valid.any():
        ax.fill_between(exp_pt[valid],
                        lo_i[valid] / exp_v0[valid],
                        hi_i[valid] / exp_v0[valid],
                        color=color, alpha=alpha, zorder=2)


def fill_ratio_band_dashed(ax, x_pred, arr, exp_v0, exp_pt, color, alpha):
    lo = np.nanpercentile(arr, 5,  axis=0)
    hi = np.nanpercentile(arr, 95, axis=0)
    lo_i = np.interp(exp_pt, x_pred, lo)
    hi_i = np.interp(exp_pt, x_pred, hi)
    valid = np.abs(exp_v0) > 1e-5
    if valid.any():
        ax.fill_between(exp_pt[valid],
                        lo_i[valid] / exp_v0[valid],
                        hi_i[valid] / exp_v0[valid],
                        color=color, alpha=alpha, zorder=2)
        ax.plot(exp_pt[valid], lo_i[valid] / exp_v0[valid],
                color=color, lw=1.5, ls='--', alpha=0.9, zorder=3)
        ax.plot(exp_pt[valid], hi_i[valid] / exp_v0[valid],
                color=color, lw=1.5, ls='--', alpha=0.9, zorder=3)


def syst_box_log(ax, xc, yc, sy, exl, exh, color, lw=1.3):
    ax.add_patch(plt.Rectangle(
        (xc - exl, yc - sy), exl + exh, 2*sy,
        lw=lw, edgecolor=color, facecolor=color, alpha=0.25, zorder=4))

# =============================================================================
# MAIN PLOT
# =============================================================================

def make_plot(pt, prior, orig, post, atlas_r, cms_r, correction, cent_label):
    fs = 18

    # Choose reference data for bottom ratio panel
    ref = atlas_r   # use ATLAS ratio as reference

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(9, 10),
        gridspec_kw={"height_ratios": [3, 1]},
        sharex=True,
    )

    # ── 1. Prior — grey hatched band ─────────────────────────────────────────
    fill_band(ax_top, pt, prior,
              color=PRIOR_COLOR, alpha=0.30,
              label="Prior 90% C.I.", hatch="///")
    if ref is not None:
        fill_ratio_band(ax_bot, pt, prior,
                        ref['v0'], ref['pT'],
                        color=PRIOR_COLOR, alpha=0.25)

    # ── 2. pT-integrated posterior — orange dashed border ────────────────────
    fill_band_dashed(ax_top, pt, orig,
                     color=POST1_COLOR, alpha=0.35,
                     label=r"Posterior 90% C.I. ($p_T$-integrated)")
    if ref is not None:
        fill_ratio_band_dashed(ax_bot, pt, orig,
                               ref['v0'], ref['pT'],
                               color=POST1_COLOR, alpha=0.30)

    # ── 3. Scaled Spectra posterior — blue solid band ─────────────────────────
    if post is not None:
        fill_band(ax_top, pt, post,
                  color=POST2_COLOR, alpha=0.40,
                  label=r"Posterior 90% C.I. (Scaled Spectra)")
        if ref is not None:
            fill_ratio_band(ax_bot, pt, post,
                            ref['v0'], ref['pT'],
                            color=POST2_COLOR, alpha=0.35)

    # ── 4. ATLAS ratio data — open circle ─────────────────────────────────────
    if atlas_r is not None:
        ax_top.errorbar(
            atlas_r['pT'], atlas_r['v0'],
            yerr=atlas_r['stat'],
            fmt='s', mfc='none', color=ATLAS_COLOR,
            markersize=8, capsize=4, lw=1.8, zorder=10,
            label=r"ATLAS 5.02 TeV  ($0.5\!-\!2$ GeV/$c$)",
        )
        for xc, yc, sy in zip(atlas_r['pT'], atlas_r['v0'], atlas_r['syst']):
            bw = xc * 0.07
            ax_top.add_patch(plt.Rectangle(
                (xc-bw, yc-sy), 2*bw, 2*sy,
                lw=1.3, edgecolor=ATLAS_COLOR, facecolor='none', zorder=11))

        # Bottom panel: ATLAS ratio/ATLAS = 1 ± err
        valid = np.abs(atlas_r['v0']) > 1e-5
        ax_bot.errorbar(
            atlas_r['pT'][valid],
            np.ones(valid.sum()),
            yerr=atlas_r['total'][valid] / atlas_r['v0'][valid],
            fmt='s', mfc='none', color=ATLAS_COLOR,
            markersize=8, capsize=4, lw=1.8, zorder=10,
        )
        for xc, sy, yv in zip(atlas_r['pT'][valid],
                               atlas_r['syst'][valid],
                               atlas_r['v0'][valid]):
            bw = xc * 0.07
            ax_bot.add_patch(plt.Rectangle(
                (xc-bw, 1.0-sy/yv), 2*bw, 2*sy/yv,
                lw=1.3, edgecolor=ATLAS_COLOR, facecolor='none', zorder=11))

    # ── 5. CMS ratio data (if available) ──────────────────────────────────────
    if cms_r is not None:
        cms_mask = cms_r['pT'] <= pt.max() * 1.1
        ax_top.errorbar(
            cms_r['pT'][cms_mask], cms_r['v0'][cms_mask],
            yerr=cms_r['stat'][cms_mask],
            fmt='s', mfc=CMS_COLOR, color=CMS_COLOR,
            markersize=8, capsize=4, lw=1.8, zorder=10,
            label=r"CMS 2.76 TeV  ($0.5\!-\!2$ GeV/$c$)",
        )
        # systematic boxes (filled, same colour, translucent)
        for xc, yc, sy in zip(cms_r['pT'][cms_mask],
                               cms_r['v0'][cms_mask],
                               cms_r['syst'][cms_mask]):
            bw = xc * 0.07
            ax_top.add_patch(plt.Rectangle(
                (xc-bw, yc-sy), 2*bw, 2*sy,
                lw=1.3, edgecolor=CMS_COLOR, facecolor=CMS_COLOR,
                alpha=0.25, zorder=11))

        # Bottom panel: CMS / ATLAS (interp ATLAS at CMS pT)
        if ref is not None:
            cms_pt  = cms_r['pT'][cms_mask]
            cms_v0  = cms_r['v0'][cms_mask]
            cms_tot = cms_r['total'][cms_mask]
            cms_syst= cms_r['syst'][cms_mask]
            ref_v0_interp = np.interp(cms_pt, ref['pT'], ref['v0'],
                                      left=np.nan, right=np.nan)
            valid_b = np.abs(ref_v0_interp) > 1e-5
            if valid_b.any():
                bot_ratio = cms_v0[valid_b] / ref_v0_interp[valid_b]
                bot_err   = cms_tot[valid_b] / ref_v0_interp[valid_b]
                bot_syst  = cms_syst[valid_b] / ref_v0_interp[valid_b]
                ax_bot.errorbar(
                    cms_pt[valid_b], bot_ratio, yerr=bot_err,
                    fmt='s', mfc=CMS_COLOR, color=CMS_COLOR,
                    markersize=8, capsize=4, lw=1.8, zorder=10,
                )
                for xc, yc, sy in zip(cms_pt[valid_b], bot_ratio, bot_syst):
                    bw = xc * 0.07
                    ax_bot.add_patch(plt.Rectangle(
                        (xc-bw, yc-sy), 2*bw, 2*sy,
                        lw=1.3, edgecolor=CMS_COLOR, facecolor=CMS_COLOR,
                        alpha=0.25, zorder=11))

    # ── Formatting ────────────────────────────────────────────────────────────
    ax_top.set_xscale('log')
    ax_top.set_xlim(pt.min()*0.80, pt.max()*1.20)
    ax_top.set_ylabel(r"$v_0(p_T)\,/\,v_0$", fontsize=fs+8)
    ax_top.axhline(1, color='k', ls='--', lw=0.8, alpha=0.4)
    ax_top.legend(loc='upper left', frameon=False, fontsize=fs-5)
    ax_top.text(0.97, 0.03,
                f"Pb+Pb,  {cent_label}\n"
                r"$p_T^{\rm ref} = 0.5\!-\!2$ GeV/$c$",
                transform=ax_top.transAxes,
                va='bottom', ha='right', fontsize=fs-4)
    configure_axis(ax_top, fs-2)

    ax_bot.axhline(1.0, color='k', ls='--', lw=1.5)
    ax_bot.set_xscale('log')
    ax_bot.set_xlim(pt.min()*0.80, pt.max()*1.20)
    ax_bot.set_ylim(0.0, 3.0)
    ax_bot.set_xlabel(r"$p_T$ [GeV/$c$]", fontsize=fs+4)
    ax_bot.set_ylabel("Model / ATLAS", fontsize=fs+2)
    ax_bot.grid(True, alpha=0.25, which='major')
    configure_axis(ax_bot, fs-2)

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.05)
    return fig

# =============================================================================
# MAIN
# =============================================================================

print("\nCreating plot...")
fig = make_plot(
    pt         = pt_values,
    prior      = prior_pred,
    orig       = orig_pred,
    post       = post_pred,
    atlas_r    = atlas_ratio,
    cms_r      = cms_ratio,
    correction = CORRECTION,
    cent_label = CENT_LABELS[CENTRALITY_INDEX],
)

label_str = CENT_LABELS[CENTRALITY_INDEX].replace('-','').replace('%','')
out_path  = Path(OUTPUT_FOLDER) / f"ratio_v0pt_v0_{CORRECTION}_{label_str}_vs_exp.pdf"
fig.savefig(out_path, dpi=300, bbox_inches='tight')
print(f"  ✓ Saved: {out_path}")
plt.close(fig)

print("\n" + "="*70)
print("✓ Done")
print("="*70 + "\n")


EMULATOR PREDICTIONS vs EXPERIMENTAL DATA  —  v0(pT)/v0
Correction: Grad  |  Centrality: 50-60%
  ✓ 17 parameters, 20000 original posterior samples
  ✓ v0(pT) emulators: 9 PCs, 13 pT bins, cent 50-60%
  ✓ v0 global obs_labels: ['0-10%', '10-20%', '20-30%', '30-40%', '40-50%', '50-60%', '60-70%', '70-80%', '80-90%', '90-100%']  →  '50-60%' = col 5
  ✓ v0 global emulators: 3 PCs

Generating predictions...
    Prior: 2000 samples ... done  shape=(2000, 13)  valid=2000  range=[-5.426, 26.549]
    Original posterior (pT-integrated): 20000 samples ... done  shape=(20000, 13)  valid=20000  range=[-4.300, 19.099]
    Scaled Spectra (Grad): 1400 samples ... done  shape=(1400, 13)  valid=1400  range=[-5.994, 17.575]
  Spread check — prior: mean σ = 1.408221
  Spread check — original: mean σ = 0.475065
  Spread check — scaled spectra: mean σ = 0.912693

Loading experimental data...
  ✓ ATLAS 5.02 TeV v0(pT)  29 bins  pT=[0.55,9.46]
  ✓ ATLAS v0 global: 9 centrality bins
  ✓ CMS 2.76 TeV v0(pT)  